# Creating Embeddings
Instantiating the Milvuslite databse and creating vector embeddings for the chatbot using the Mines registrar, catalog, CASA, and department websites.

## Creating the Milvus Lite Database

In [28]:
from pymilvus import MilvusClient

milvus_client = MilvusClient("store.db")

In [29]:
if milvus_client.has_collection(collection_name="chatbot"): # NOTE: if you are running this notebook in full
    milvus_client.drop_collection(collection_name="chatbot")
milvus_client.create_collection(
    collection_name="chatbot",
    dimension=1536 # text-embedding-3-small
)

## Creating Document Embeddings

In [30]:
from openai import OpenAI
from dotenv import load_dotenv
from os import environ

load_dotenv()

openai_client = OpenAI(api_key=environ["OPENAI-API-KEY"])

## Chunking documents

In [31]:
import json
import re
from pathlib import Path

import tiktoken

DOCS_DIR = Path("documents")
ENC = tiktoken.get_encoding("cl100k_base")  # tokenizer used by text-embedding-3-small

TARGET_TOKENS = 500   # small section merge target
MAX_TOKENS = 800      # sections exceeding MAX_TOKENS will be split
OVERLAP_TOKENS = 60   # amount of tokens to overlap between sections
EMBED_LIMIT = 8191    # text-embedding-3-small input limit

HEADING_RE = re.compile(r"^(#{1,6})\s+(\S.*)$")
EMPTY_HEADING_RE = re.compile(r"^#{1,6}\s*$", re.M)
COURSE_RE = re.compile(r"^([A-Z]{4}\d{3}[A-Z]?)\. (.+?)\. ([\d.\-]+) Semester Hrs?\.", re.M)

SITE_NAMES = {
    "registrar.mines.edu": "Registrar",
    "casa.mines.edu": "CASA",
    "graduatestudies.mines.edu": "Graduate Studies",
    "catalog.mines.edu": "Catalog",
    "www.mines.edu": "Mines",
}


def n_tokens(text: str) -> int:
    return len(ENC.encode(text))


def site_name(host: str) -> str:
    return SITE_NAMES.get(host, host.split(".")[0].upper() if len(host.split(".")[0]) <= 4
                          else host.split(".")[0].title())


def doc_type(doc: dict) -> str:
    url, host = doc["url"], doc["source_site"]
    if "/coursesaz/" in url and not url.endswith("/coursesaz/"):
        return "course"
    if host == "casa.mines.edu":
        return "advising"
    if host in ("registrar.mines.edu", "graduatestudies.mines.edu"):
        return "policy"
    if host == "catalog.mines.edu":
        return "program" if re.search(r"/(degrees|minors|programs)/", url) else "policy"
    return "program"


def clean(text: str) -> str:
    """Drop empty headings (lines that are only #'s) and extra blank lines."""
    text = EMPTY_HEADING_RE.sub("", text)
    return re.sub(r"\n{3,}", "\n\n", text).strip()

### Chunking: split at headings, merge small sections, split long ones

In [32]:
def split_sections(text: str) -> list[tuple[list[str], str]]:
    """Split at markdown headings. Returns (heading_path, body) pairs; body keeps its heading line.
    A heading with no text under it is carried into the next section."""
    out, path, buf = [], [], []

    def flush() -> bool:
        if not any(l.strip() and not HEADING_RE.match(l) for l in buf):
            return False  # headings only: keep them for the next section
        out.append(([t for _, t in path], "\n".join(buf).strip()))
        return True

    for line in text.split("\n"):
        m = HEADING_RE.match(line)
        if m:
            if flush():
                buf = []
            level = len(m.group(1))
            path = [p for p in path if p[0] < level] + [(level, m.group(2).strip())]
        buf.append(line)
    flush()
    return out


def _units(text: str) -> list[str]:
    """Standardizes paragraphs, then lines for oversized paragraphs, then token windows for oversized lines.
    Lines are never cut unless a single line exceeds MAX_TOKENS."""
    units = []
    for para in re.split(r"\n\s*\n", text):
        if n_tokens(para) <= MAX_TOKENS:
            units.append(para)
            continue
        for line in para.split("\n"):
            toks = ENC.encode(line)
            if len(toks) <= MAX_TOKENS:
                units.append(line)
            else:
                units += [ENC.decode(toks[i:i + TARGET_TOKENS]) for i in range(0, len(toks), TARGET_TOKENS)]
    return [u for u in units if u.strip()]


def split_long(text: str) -> list[str]:
    """Pack units into ~TARGET_TOKENS pieces with ~OVERLAP_TOKENS of overlap between pieces."""
    pieces, cur, cur_tok = [], [], 0
    for unit in _units(text):
        t = n_tokens(unit)
        if cur and cur_tok + t > TARGET_TOKENS:
            pieces.append("\n\n".join(cur))
            tail, tail_tok = [], 0           # carry trailing units forward as overlap
            for u in reversed(cur):
                ut = n_tokens(u)
                if tail_tok + ut > OVERLAP_TOKENS:
                    break
                tail.insert(0, u)
                tail_tok += ut
            cur, cur_tok = tail, tail_tok
        cur.append(unit)
        cur_tok += t
    if cur:
        pieces.append("\n\n".join(cur))
    return pieces


def merge_sections(sections: list[tuple[list[str], str]]) -> list[tuple[list[str], str]]:
    """Merge neighbouring small sections that share a top-level heading and split oversized sections."""
    out, group, group_tok = [], [], 0

    def common_path(paths):
        prefix = paths[0]
        for p in paths[1:]:
            i = 0
            while i < min(len(prefix), len(p)) and prefix[i] == p[i]:
                i += 1
            prefix = prefix[:i]
        return prefix

    def flush():
        if group:
            out.append((common_path([p for p, _ in group]), "\n\n".join(b for _, b in group)))

    for path, body in sections:
        t = n_tokens(body)
        if t > MAX_TOKENS:
            flush(); group, group_tok = [], 0
            out += [(path, piece) for piece in split_long(body)]
            continue
        same_parent = group and group[0][0][:1] == path[:1]
        if group and (not same_parent or group_tok + t > TARGET_TOKENS):
            flush(); group, group_tok = [], 0
        group.append((path, body))
        group_tok += t
    flush()
    return out

### Chunking: one document (course pages split per course)

In [33]:
def chunk_document(doc: dict) -> list[dict]:
    text = clean(doc["text"])
    dtype = doc_type(doc)
    pieces = []  # (heading_path, body, course_code, credits)

    if dtype == "course":
        matches = list(COURSE_RE.finditer(text))
        if matches and matches[0].start() > 0:
            pre = text[:matches[0].start()].strip()
            pieces += [(p, b, "", "") for p, b in merge_sections(split_sections(pre))]
        for i, m in enumerate(matches):
            end = matches[i + 1].start() if i + 1 < len(matches) else len(text)
            body = text[m.start():end].strip()
            path = [f"{m.group(1)} {m.group(2).title()}"]
            for part in (split_long(body) if n_tokens(body) > MAX_TOKENS else [body]):
                pieces.append((path, part, m.group(1), m.group(3)))
        if not matches:
            pieces += [(p, b, "", "") for p, b in merge_sections(split_sections(text))]
    else:
        pieces = [(p, b, "", "") for p, b in merge_sections(split_sections(text))]

    chunks = []
    for i, (path, body, code, credits) in enumerate(pieces):
        crumbs = [site_name(doc["source_site"]), doc["title"]]
        crumbs += [h for h in path if h.lower() != doc["title"].lower()]
        text_out = f"{body}"
        chunks.append({
            "text": text_out,
            "url": doc["url"],
            "title": doc["title"],
            "source_site": doc["source_site"],
            "priority": doc["priority"],
            "modified": doc["modified"],
            "heading_path": " > ".join(path),
            "chunk_index": i,
            "doc_type": dtype,
            "course_code": code,
            "credits": credits,
            "n_tokens": n_tokens(text_out),
        })
    return chunks

### Chunking: the whole corpus

In [34]:
docs = [json.loads(p.read_text()) for p in sorted(DOCS_DIR.glob("*.json"))]
chunks = [c for d in docs for c in chunk_document(d)]
for i, c in enumerate(chunks):
    c["id"] = i

sizes = sorted(c["n_tokens"] for c in chunks)
print(f"{len(docs)} documents -> {len(chunks)} chunks")
print(f"tokens per chunk: median {sizes[len(sizes) // 2]}, p90 {sizes[int(len(sizes) * .9)]}, max {sizes[-1]}")
print(f"total tokens to embed: {sum(sizes):,}")
from collections import Counter
print(Counter(c["doc_type"] for c in chunks))

340 documents -> 5848 chunks
tokens per chunk: median 190, p90 489, max 814
total tokens to embed: 1,384,362
Counter({'course': 2901, 'program': 2064, 'policy': 751, 'advising': 132})


## Store the Chunks into the Database

In [35]:
texts = [obj["text"] for obj in chunks]

if len(chunks) != len(texts):
    raise RuntimeError("The amount of chunk text fields must equal the amount of chunks.")

total = len(texts)
vectors = []
for i, text in enumerate(texts, start=1): # this loop will take >5m
    vectors.append(openai_client.embeddings.create(
        input=text, model="text-embedding-3-small"
    ))
    print(f"Embedded {i}/{total} chunks ({i / total:.1%})", end="\r")
print()

I1112 17:38:29.594017 17774269 chttp2_transport.cc:1403] ipv4:127.0.0.1:65193: Got goaway [11] err=UNAVAILABLE:GOAWAY received; Error code: 11; Debug Text: too_many_pings {http2_error:11}
E1112 17:38:29.594831 17774269 chttp2_transport.cc:1435] ipv4:127.0.0.1:65193: Received a GOAWAY with error code ENHANCE_YOUR_CALM and debug data equal to "too_many_pings". Current keepalive time (before throttling): 10000ms


Embedded 5848/5848 chunks (100.0%)


In [37]:
if len(chunks) != len(vectors):
    raise RuntimeError("The amount of chunk embeddings must equal the amount of chunks.")

data = [
    {
        "id": i,
        "chunk_index": c["chunk_index"],
        "vector": vectors[i].data[0].embedding,
        "chunk": c["text"],
        "url": c["url"],
        "title": c["title"],
        "source_site": c["source_site"],
        "priority": c["priority"],
        "modified": c["modified"],
        "heading_path": c["heading_path"],
        "doc_type": c["doc_type"],
        "course_code": c["course_code"],
        "credits": c["credits"],
        "n_tokens": c["n_tokens"]
    }
    for i, c in enumerate(chunks)
]
milvus_client.insert(collection_name="chatbot", data=data)
milvus_client.close()